# H-Copilot Flow Prediction — Clean Final Build
This notebook rebuilds the final 15-feature blended model (Random Forest + Neural Network) directly, skipping all the rejected experiments. Upload `ED_triage.csv` when prompted, then Run All.

## 1. Imports and upload data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload ED_triage.csv when prompted

Saving ED_triage.csv to ED_triage.csv


## 2. Load and clean raw data

In [ ]:
triage = pd.read_csv('ED_triage.csv')

# Remove duplicate triage_code rows (re-triage records of same visit), keep first
triage = triage.drop_duplicates(subset='triage_code', keep='first')

# Build proper date column
triage['date'] = pd.to_datetime(
    triage[['admission_year','admission_month','admission_day']]
    .rename(columns={'admission_year':'year','admission_month':'month','admission_day':'day'})
)

# day_of_week: 0=Sunday ... 6=Saturday (verified against real calendar earlier in project)
triage['day_of_week'] = (triage['date'].dt.dayofweek + 1) % 7

# Build hour_slot (0-5) from admission_hour — not pre-existing in the raw CSV
triage['hour_slot'] = triage['admission_hour'] // 4

print(f'Rows after dedup: {len(triage)}')
print(triage[['date','day_of_week','hour_slot']].head())

Rows after dedup: 143346
        date  day_of_week  hour_slot
0 2017-03-21            2          0
1 2017-03-21            2          0
2 2017-03-21            2          0
3 2017-03-21            2          0
4 2017-03-21            2          0


## 3. Keep only the clean, COVID-free continuous range
Data confirmed normal through Feb 21, 2020 (Iran's first COVID cases announced Feb 19, 2020; sharp arrival drop began Feb 22, 2020).

In [ ]:
triage = triage[triage['date'] <= '2020-02-21'].copy()
print(f'Rows in clean range: {len(triage)}')
print(f'Date range: {triage["date"].min()} to {triage["date"].max()}')

Rows in clean range: 91490
Date range: 2017-03-21 00:00:00 to 2020-02-21 00:00:00


## 4. Build the slot-level patient_count table (the base for flow prediction)

In [ ]:
base = (
    triage.groupby(['date','day_of_week','hour_slot'])
    .size()
    .reset_index(name='patient_count')
)
base = base.sort_values(['date','hour_slot']).reset_index(drop=True)
print(base.head())
print(f'Total slot-rows: {len(base)}')

        date  day_of_week  hour_slot  patient_count
0 2017-03-21            2          0              8
1 2017-03-21            2          1             11
2 2017-03-21            2          2             16
3 2017-03-21            2          3             21
4 2017-03-21            2          4             26
Total slot-rows: 6367


## 5. Build all 15 final features
Each one individually tested and validated earlier in the project.

In [ ]:
# --- Historical patient-flow features ---
# Each weekday + 4-hour slot occurs approximately once per week.
# Therefore, rolling N observations represent approximately N previous weeks.

for weeks in [2, 4, 8, 12]:
    base[f'lag_{weeks}wk_avg'] = (
        base.groupby(['day_of_week', 'hour_slot'])['patient_count']
        .transform(
            lambda x: x.shift(1).rolling(
                weeks,
                min_periods=1
            ).mean()
        )
    )

# --- Same-day momentum ---
base['daily_cumulative_avg'] = base.groupby('date')['patient_count'].transform(lambda x: x.expanding().mean().shift(1))
base['daily_cumulative_avg'] = base['daily_cumulative_avg'].fillna(base['lag_8wk_avg'])

# --- Calendar ---
base['is_weekend'] = base['day_of_week'].isin([0,6]).astype(int)
base['month'] = pd.to_datetime(base['date']).dt.month
base['week_of_year'] = pd.to_datetime(base['date']).dt.isocalendar().week.astype(int)

def add_slot_feature(df, source_col, agg='mean', newname=None):
    name = newname or f'avg_{source_col}'
    agg_df = (
        triage.groupby(['date','hour_slot'])[source_col]
        .agg(agg)
        .reset_index()
        .rename(columns={source_col: name})
    )
    df = df.merge(agg_df, on=['date','hour_slot'], how='left')
    lagname = f'lag_8wk_{name}'
    df[lagname] = df.groupby(['day_of_week','hour_slot'])[name].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
    return df, lagname

base, sev_col = add_slot_feature(base, 'TriageGrade', newname='severity')
base, age_col = add_slot_feature(base, 'age')
base, pain_col = add_slot_feature(base, 'PainGrade')
base, fast_col = add_slot_feature(base, 'NeedFastExecute')
base, crit_col = add_slot_feature(base, 'CriticalStatus')

# --- High-acuity rate (Grade 1-2) ---
triage['is_high_acuity'] = (triage['TriageGrade'] <= 2).astype(int)
base, acuity_col = add_slot_feature(base, 'is_high_acuity', newname='high_acuity')

# --- Complaint-type mix (4 macro categories from ICD chapter letters) ---
triage['icd_chapter'] = triage['ChiefComplaint'].astype(str).str[0]
category_map = {
    'S':'Trauma/Injury','T':'Trauma/Injury','W':'Trauma/Injury',
    'R':'Medical/Internal','K':'Medical/Internal','N':'Medical/Internal',
    'I':'Medical/Internal','G':'Medical/Internal','E':'Medical/Internal','J':'Medical/Internal',
    'M':'Musculoskeletal/Skin','L':'Musculoskeletal/Skin',
    'Z':'Other/Administrative'
}
triage['complaint_macro'] = triage['icd_chapter'].map(category_map).fillna('Other/Administrative')

complaint_mix = (
    triage.groupby(['date','hour_slot','complaint_macro'])
    .size().unstack(fill_value=0).reset_index()
)
base = base.merge(complaint_mix, on=['date','hour_slot'], how='left')
for cat in ['Medical/Internal','Trauma/Injury']:  # the 2 categories kept in the final 15-feature set
    base[f'lag_8wk_{cat}'] = base.groupby(['day_of_week','hour_slot'])[cat].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())

base = base.sort_values(['date','hour_slot']).reset_index(drop=True)
print('Feature build complete.')
print(base.columns.tolist())


Feature build complete.
['date', 'day_of_week', 'hour_slot', 'patient_count', 'lag_2wk_avg', 'lag_4wk_avg', 'lag_8wk_avg', 'lag_12wk_avg', 'daily_cumulative_avg', 'is_weekend', 'month', 'week_of_year', 'severity', 'lag_8wk_severity', 'avg_age', 'lag_8wk_avg_age', 'avg_PainGrade', 'lag_8wk_avg_PainGrade', 'avg_NeedFastExecute', 'lag_8wk_avg_NeedFastExecute', 'avg_CriticalStatus', 'lag_8wk_avg_CriticalStatus', 'high_acuity', 'lag_8wk_high_acuity', 'Medical/Internal', 'Musculoskeletal/Skin', 'Other/Administrative', 'Trauma/Injury', 'lag_8wk_Medical/Internal', 'lag_8wk_Trauma/Injury']


In [ ]:
import numpy as np, pandas as pd, time, warnings; warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

FEATS = ['hour_slot','day_of_week','month','week_of_year','is_weekend',
         'lag_2wk_avg','lag_4wk_avg','lag_8wk_avg','lag_12wk_avg',
         'lag_8wk_severity','lag_8wk_high_acuity','daily_cumulative_avg',
         'lag_8wk_avg_PainGrade','lag_8wk_Medical/Internal','lag_8wk_Trauma/Injury']
src = base.copy() if 'base' in globals() else pd.read_csv('historical_flow_raw.csv', parse_dates=['date'])
src['date'] = pd.to_datetime(src['date'])
df = src[src.date <= '2020-02-21'].sort_values(['date', 'hour_slot']).dropna(subset=FEATS).reset_index(drop=True)
n = len(df); start = int(n * 0.8)
print(f'rows {n} | train {start} | test {n-start}   (thesis: 6325 | 5060 | 1265)')

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); nz = y != 0; pk = y >= np.quantile(y, .9)
    return dict(MAE=round(np.abs(y-p).mean(), 3), RMSE=round(np.sqrt(((y-p)**2).mean()), 3), MAPE=round(100*np.mean(np.abs((y[nz]-p[nz])/y[nz])), 2),
                WAPE=round(100*np.abs(y-p).sum()/y.sum(), 2), PeakMAE=round(np.abs(y[pk]-p[pk]).mean(), 3))

def blend_fit_predict(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr)
    rf = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42, n_jobs=-1).fit(Xtr, ytr)
    m = MLPRegressor(hidden_layer_sizes=(50,), alpha=1e-4, learning_rate_init=0.01, max_iter=300, random_state=42,
                     early_stopping=True, n_iter_no_change=5).fit(sc.transform(Xtr), ytr)
    return (rf.predict(Xte) + m.predict(sc.transform(Xte))) / 2

def walk_forward_blend(feats, frac=0.8, step=126):
    s = int(n * frac); ys, ps, i = [], [], s
    while i < n:
        j = min(i + step, n)
        ps += list(blend_fit_predict(df.loc[:i-1, feats], df.loc[:i-1, 'patient_count'], df.loc[i:j-1, feats]))
        ys += list(df.loc[i:j-1, 'patient_count']); i = j
    return np.array(ys), np.array(ps)

y_te = df.loc[start:, 'patient_count'].values
naive = df.loc[start:, 'lag_4wk_avg'].values                       # naive baseline = average of the last 4 same slots
if 'preds' in globals() and 'RF + MLP Blend (final)' in preds and len(preds['RF + MLP Blend (final)']) == len(y_te):
    wf = np.asarray(preds['RF + MLP Blend (final)']); print('re-using the blend walk forward predictions from the big comparison cell')
else:
    t = time.time(); _, wf = walk_forward_blend(FEATS); print(f'walk forward blend computed in {time.time()-t:.0f}s')
print('blend  ', metrics(y_te, wf), '  <- thesis: MAE 3.09, RMSE 3.95, MAPE 30.97, WAPE 20.70, Peak 6.15')
print('naive  ', metrics(y_te, naive), '  <- thesis: MAE 3.29, RMSE 4.19, MAPE 31.83')


rows 6325 | train 5060 | test 1265   (thesis: 6325 | 5060 | 1265)
walk forward blend computed in 1286s
blend   {'MAE': np.float64(3.089), 'RMSE': np.float64(3.948), 'MAPE': np.float64(30.97), 'WAPE': np.float64(20.62), 'PeakMAE': np.float64(6.153)}   <- thesis: MAE 3.09, RMSE 3.95, MAPE 30.97, WAPE 20.70, Peak 6.15
naive   {'MAE': np.float64(3.293), 'RMSE': np.float64(4.194), 'MAPE': np.float64(31.83), 'WAPE': np.float64(21.98), 'PeakMAE': np.float64(5.37)}   <- thesis: MAE 3.29, RMSE 4.19, MAPE 31.83


In [ ]:
rng = np.random.default_rng(42)
d = np.abs(y_te - naive) - np.abs(y_te - wf)                 # positive = the blend is closer to the truth than the naive baseline
obs = d.mean()
blocks = np.arange(len(d)) // 6; nb = blocks.max() + 1
null = np.array([(d * rng.choice([-1, 1], nb)[blocks]).mean() for _ in range(10000)])
p_perm = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f'observed improvement in MAE (naive - blend): {obs:.3f} patients per slot')
print(f'sign flip permutation test, 10,000 flips: p = {p_perm:.4f}   (thesis: 0.0068)')
bs = np.array([rng.choice(d, len(d)).mean() for _ in range(5000)])
print(f'paired bootstrap 95% CI of the improvement: {np.percentile(bs, 2.5):.3f} to {np.percentile(bs, 97.5):.3f}')
print('NOTE: with exactly 1,000 permutations p cannot go below about 0.001, so 10,000 flips are used here.')
print('      The result is reported as p < 0.001.')

observed improvement in MAE (naive - blend): 0.203 patients per slot
sign flip permutation test, 10,000 flips: p = 0.0001   (thesis: 0.0068)
paired bootstrap 95% CI of the improvement: 0.115 to 0.289
NOTE: with exactly 1,000 permutations p cannot go below about 0.001, so 10,000 flips are used here.
      The result is reported as p < 0.001.


In [ ]:
cvdata = df.iloc[:start].reset_index(drop=True)
res_cv = []
for k, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5).split(cvdata), 1):
    a, b = cvdata.iloc[tr], cvdata.iloc[te]
    rf = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42, n_jobs=-1).fit(a[FEATS], a['patient_count'])
    m = MLPRegressor(hidden_layer_sizes=(50,), alpha=1e-4, learning_rate_init=0.01, max_iter=2000, random_state=42).fit(a[FEATS], a['patient_count'])
    e = mean_absolute_error(b['patient_count'], (rf.predict(b[FEATS]) + m.predict(b[FEATS])) / 2); res_cv.append(e)
    print(f'Fold {k}: MAE {e:.2f}  (train {len(a)}, test {len(b)})')
print(f'range {min(res_cv):.2f} to {max(res_cv):.2f} | mean {np.mean(res_cv):.2f} | SD {np.std(res_cv):.2f}   (thesis: 3.02 to 3.32, SD 0.12)')
res_cv2 = []
for k, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5).split(cvdata), 1):    # same CV with the SAME model as the final blend (scaled MLP + early stopping)
    a, b = cvdata.iloc[tr], cvdata.iloc[te]
    res_cv2.append(mean_absolute_error(b['patient_count'], blend_fit_predict(a[FEATS], a['patient_count'], b[FEATS])))
print('CV with the final blend setup (scaled MLP):', [round(x, 2) for x in res_cv2], f'| range {min(res_cv2):.2f} to {max(res_cv2):.2f} | SD {np.std(res_cv2):.2f}')

Fold 1: MAE 3.07  (train 845, test 843)
Fold 2: MAE 3.02  (train 1688, test 843)
Fold 3: MAE 3.05  (train 2531, test 843)
Fold 4: MAE 3.22  (train 3374, test 843)
Fold 5: MAE 3.32  (train 4217, test 843)
range 3.02 to 3.32 | mean 3.14 | SD 0.12   (thesis: 3.02 to 3.32, SD 0.12)
CV with the final blend setup (scaled MLP): [4.12, 3.14, 3.06, 3.21, 3.34] | range 3.06 to 4.12 | SD 0.39


In [ ]:
class BlendModel(RegressorMixin, BaseEstimator):
    def fit(self, X, y): self.X_, self.y_ = X, y; self.sc = StandardScaler().fit(X); self.rf = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42, n_jobs=-1).fit(X, y); self.mlp = MLPRegressor(hidden_layer_sizes=(50,), alpha=1e-4, learning_rate_init=0.01, max_iter=300, random_state=42, early_stopping=True, n_iter_no_change=5).fit(self.sc.transform(X), y); return self
    def predict(self, X): return (self.rf.predict(X) + self.mlp.predict(self.sc.transform(X))) / 2
bm = BlendModel().fit(df.loc[:start-1, FEATS], df.loc[:start-1, 'patient_count'])
pi = permutation_importance(bm, df.loc[start:, FEATS], y_te, scoring='neg_mean_absolute_error', n_repeats=20, random_state=42)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
share = (100 * imp.clip(lower=0) / imp.clip(lower=0).sum()).round(1)
print(pd.DataFrame({'importance (MAE rise)': imp.round(3), 'share % of positive': share}).to_string())
pos = [f for f in FEATS if pi.importances_mean[FEATS.index(f)] > 0]
print(f'\nfeatures with positive importance: {len(pos)} of 15 ->', pos)
y_sub, p_sub = walk_forward_blend(pos)
print('FULL 15 features :', metrics(y_te, wf), ' (thesis 3.09 / 30.97 / 6.15)')
print('POSITIVE subset  :', metrics(y_sub, p_sub), ' (thesis 3.09 / 29.89 / 6.47)')

                          importance (MAE rise)  share % of positive
lag_12wk_avg                              1.649                 67.8
lag_8wk_avg                               0.307                 12.6
hour_slot                                 0.299                 12.3
lag_2wk_avg                               0.063                  2.6
lag_8wk_Medical/Internal                  0.044                  1.8
lag_4wk_avg                               0.026                  1.1
lag_8wk_high_acuity                       0.017                  0.7
day_of_week                               0.011                  0.5
daily_cumulative_avg                      0.010                  0.4
is_weekend                                0.007                  0.3
lag_8wk_Trauma/Injury                     0.001                  0.0
lag_8wk_severity                         -0.003                  0.0
week_of_year                             -0.006                  0.0
lag_8wk_avg_PainGrade             

In [ ]:
e = wf - y_te
print(f'within 3 patients: {100*np.mean(np.abs(e) <= 3):.0f}%   within 5: {100*np.mean(np.abs(e) <= 5):.0f}%   (thesis 58% / 81%)')
print(f'prediction range: {wf.min():.1f} to {wf.max():.1f}   observed max: {y_te.max()}   (thesis about 4 to 24, 35)')
top = y_te >= np.quantile(y_te, .9); bot = y_te <= np.quantile(y_te, .1)
print(f'busiest tenth: {top.sum()} slots, forecast below observed in {np.sum(e[top] < 0)} of them, mean shortfall {-e[top].mean():.2f}   (thesis 133 slots, all but one, 6.15)')
print(f'quietest tenth: {bot.sum()} slots, mean overshoot {e[bot].mean():.2f}   (thesis about 2.4)')

within 3 patients: 58%   within 5: 81%   (thesis 58% / 81%)
prediction range: 3.7 to 24.2   observed max: 35   (thesis about 4 to 24, 35)
busiest tenth: 133 slots, forecast below observed in 132 of them, mean shortfall 6.15   (thesis 133 slots, all but one, 6.15)
quietest tenth: 136 slots, mean overshoot 2.38   (thesis about 2.4)


In [ ]:
y70, p70 = walk_forward_blend(FEATS, frac=0.7)
print('70/30 walk forward :', metrics(y70, p70), f'  (test rows {len(y70)})')
print('80/20 walk forward :', metrics(y_te, wf))

70/30 walk forward : {'MAE': np.float64(3.151), 'RMSE': np.float64(4.047), 'MAPE': np.float64(30.39), 'WAPE': np.float64(21.0), 'PeakMAE': np.float64(6.678)}   (test rows 1898)
80/20 walk forward : {'MAE': np.float64(3.089), 'RMSE': np.float64(3.948), 'MAPE': np.float64(30.97), 'WAPE': np.float64(20.62), 'PeakMAE': np.float64(6.153)}


In [ ]:
clean_features = base.copy()

In [ ]:
# ===================== H-Copilot: re-run ALL model comparisons in ONE cell =====================
# Same protocol as the thesis: chronological 80/20 split, walk forward, retrain every 126 slots, 15 features.
# Safe for overnight: every model is wrapped in try/except and results are saved to disk after each model.
import subprocess, sys, time, warnings, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'catboost', 'statsmodels'])
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
try:
    from lightgbm import LGBMRegressor
except Exception: LGBMRegressor = None
try:
    from catboost import CatBoostRegressor
except Exception: CatBoostRegressor = None
try:
    import statsmodels.api as sm
except Exception: sm = None

TUNE = True            # tune LightGBM / CatBoost / SVR / KNN / ElasticNet / Neural Net on the TRAINING part only (TimeSeriesSplit)
CSV_PATH = 'historical_flow_raw.csv'   # used only if the variable clean_features does not exist in this notebook
OUT = 'model_comparison_rerun.csv'

FEATS = ['hour_slot','day_of_week','month','week_of_year','is_weekend',
         'lag_2wk_avg','lag_4wk_avg','lag_8wk_avg','lag_12wk_avg',
         'lag_8wk_severity','lag_8wk_high_acuity','daily_cumulative_avg',
         'lag_8wk_avg_PainGrade','lag_8wk_Medical/Internal','lag_8wk_Trauma/Injury']
TARGET, STEP, TRAIN_FRAC = 'patient_count', 126, 0.8

# ---------- data ----------
raw = clean_features.copy() if 'clean_features' in globals() else pd.read_csv(CSV_PATH, parse_dates=['date'])
raw['date'] = pd.to_datetime(raw['date'])
raw = raw[raw.date <= '2020-02-21']                         # clean pre-COVID range (also drops the stray 2026 platform row)
df = raw.sort_values(['date', 'hour_slot']).dropna(subset=FEATS).reset_index(drop=True)
n = len(df); start = int(n * TRAIN_FRAC)
print(f'rows {n} | train {start} | test {n-start}   (thesis: 6325 | 5060 | 1265)')
Xtr_all, ytr_all = df.loc[:start-1, FEATS], df.loc[:start-1, TARGET]

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); nz = y != 0; pk = y >= np.quantile(y, .9)
    return dict(MAE=np.abs(y-p).mean(), RMSE=np.sqrt(((y-p)**2).mean()), MAPE=100*np.mean(np.abs((y[nz]-p[nz])/y[nz])),
                WAPE=100*np.abs(y-p).sum()/y.sum(), PeakMAE=np.abs(y[pk]-p[pk]).mean())

# ---------- model builders ----------
def rf():  return RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42, n_jobs=-1)
def mlp(**kw):
    p = dict(hidden_layer_sizes=(50,), alpha=1e-4, learning_rate_init=0.01, max_iter=300, random_state=42, early_stopping=True, n_iter_no_change=5); p.update(kw)
    return MLPRegressor(**p)
class Scaled:                                   # neural network on standardized inputs
    def __init__(s, **kw): s.kw = kw
    def fit(s, X, y): s.sc = StandardScaler().fit(X); s.m = mlp(**s.kw).fit(s.sc.transform(X), y); return s
    def predict(s, X): return s.m.predict(s.sc.transform(X))
class Blend:                                    # final model: 50/50 RF + MLP
    def fit(s, X, y): s.sc = StandardScaler().fit(X); s.rf = rf().fit(X, y); s.m = mlp().fit(s.sc.transform(X), y); return s
    def predict(s, X): return (s.rf.predict(X) + s.m.predict(s.sc.transform(X))) / 2
class Col:                                      # "no model": just copy one feature column
    def __init__(s, col): s.col = col
    def fit(s, X, y): return s
    def predict(s, X): return X[s.col].values

def tuned(pipe_or_est, grid, name):
    if not TUNE: return pipe_or_est
    t = time.time()
    gs = GridSearchCV(pipe_or_est, grid, cv=TimeSeriesSplit(5), scoring='neg_mean_absolute_error', n_jobs=-1).fit(Xtr_all, ytr_all)
    print(f'   tuned {name}: {gs.best_params_}  ({time.time()-t:.0f}s)')
    return gs.best_estimator_

def walk_forward(make):
    ys, ps, i = [], [], start
    while i < n:                                # train on everything before i, predict the next 126 slots, slide forward
        j = min(i + STEP, n)
        m = make().fit(df.loc[:i-1, FEATS], df.loc[:i-1, TARGET])
        ps += list(m.predict(df.loc[i:j-1, FEATS])); ys += list(df.loc[i:j-1, TARGET]); i = j
    return np.array(ys), np.array(ps)

from sklearn.base import clone
def make_tuned(est_factory, grid, name):
    best = tuned(est_factory(), grid, name)
    return lambda: clone(best)

results, preds, y_test = [], {}, None
def record(name, y, p, note=''):
    r = {'Model': name, **metrics(y, p), 'Note': note}; results.append(r)
    pd.DataFrame(results).round(3).to_csv(OUT, index=False)          # saved after every model
    print(f'>> {name:34s} MAE {r["MAE"]:.3f}  RMSE {r["RMSE"]:.3f}  MAPE {r["MAPE"]:.2f}  WAPE {r["WAPE"]:.2f}  Peak {r["PeakMAE"]:.2f}', flush=True)

def run(name, make, note=''):
    global y_test
    t = time.time()
    try:
        y, p = walk_forward(make); y_test = y; preds[name] = p; record(name, y, p, note)
    except Exception as e:
        print(f'!! {name} FAILED: {e}'); results.append({'Model': name, 'Note': f'FAILED: {e}'})
    print(f'   ({time.time()-t:.0f}s)', flush=True)

# ---------- 1. baselines with no learning ----------
run('Naive baseline (avg of last 4 same slots)', lambda: Col('lag_4wk_avg'), 'lag_4wk_avg')
run('Reference: avg of last 12 same slots', lambda: Col('lag_12wk_avg'), 'lag_12wk_avg, no model')

# ---------- 2. machine learning models, walk forward ----------
run('Random Forest (alone)', rf, 'final settings')
make_nn = make_tuned(lambda: make_pipeline(StandardScaler(), MLPRegressor(max_iter=300, random_state=42, early_stopping=True, n_iter_no_change=5)),
                     {'mlpregressor__hidden_layer_sizes': [(25,), (50,), (100,)], 'mlpregressor__alpha': [1e-4, 1e-3, 1e-2], 'mlpregressor__learning_rate_init': [1e-3, 1e-2]}, 'Neural Network') if TUNE else (lambda: Scaled())
run('Neural Network (alone, tuned)' if TUNE else 'Neural Network (alone)', make_nn)
if LGBMRegressor:
    run('LightGBM (tuned)' if TUNE else 'LightGBM', make_tuned(lambda: LGBMRegressor(random_state=42, verbose=-1),
        {'num_leaves': [15, 31], 'n_estimators': [200, 400], 'learning_rate': [0.03, 0.1], 'max_depth': [-1, 6]}, 'LightGBM'))
else: print('LightGBM not available')
if CatBoostRegressor:
    run('CatBoost (tuned)' if TUNE else 'CatBoost', make_tuned(lambda: CatBoostRegressor(random_seed=42, verbose=0, thread_count=-1),
        {'depth': [4, 6, 8], 'learning_rate': [0.03, 0.1], 'iterations': [300]}, 'CatBoost'))
else: print('CatBoost not available')
run('Support Vector Regression', make_tuned(lambda: make_pipeline(StandardScaler(), SVR()),
    {'svr__C': [1, 10, 100], 'svr__epsilon': [0.1, 1.0]}, 'SVR'))
run('K Nearest Neighbors (tuned)', make_tuned(lambda: make_pipeline(StandardScaler(), KNeighborsRegressor()),
    {'kneighborsregressor__n_neighbors': [5, 10, 15, 25, 40], 'kneighborsregressor__weights': ['uniform', 'distance']}, 'KNN'))
run('Elastic Net (linear)', make_tuned(lambda: make_pipeline(StandardScaler(), ElasticNet(random_state=42)),
    {'elasticnet__alpha': [0.01, 0.1, 1.0], 'elasticnet__l1_ratio': [0.2, 0.5, 0.8]}, 'ElasticNet'))
run('RF + MLP Blend (final)', Blend, 'the thesis model')

# ---------- 3. classical baselines, as in the original cells (fit once, no walk forward) ----------
try:
    if sm is None: raise RuntimeError('statsmodels not installed')
    # SARIMAX: one fit on the training part, forecast the whole test period
    y_full = df[TARGET].values; y_tr, y_te = y_full[:start], y_full[start:]
    fit = sm.tsa.statespace.SARIMAX(y_tr, order=(1,0,1), seasonal_order=(1,0,1,6), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    record('SARIMAX (fit once, long horizon)', y_te, fit.forecast(steps=len(y_te)), 'order (1,0,1)(1,0,1,6), no exogenous')
except Exception as e: print('!! SARIMAX FAILED', e)
try:                                            # NNAR style: MLP on the previous 6 raw slot values, trained once
    d2 = df.sort_values(['date', 'hour_slot']).reset_index(drop=True)
    for k in range(1, 7): d2[f'raw_lag_{k}'] = d2[TARGET].shift(k)
    lagc = [f'raw_lag_{k}' for k in range(1, 7)]; d2 = d2.dropna(subset=lagc); s2 = int(len(d2) * 0.8)
    m = MLPRegressor(hidden_layer_sizes=(50,), max_iter=2000, random_state=42).fit(d2.iloc[:s2][lagc], d2.iloc[:s2][TARGET])
    record('NNAR style MLP (raw 6 lags)', d2.iloc[s2:][TARGET], m.predict(d2.iloc[s2:][lagc]), f'test rows {len(d2)-s2}, trained once')
except Exception as e: print('!! NNAR FAILED', e)

# ---------- 4. summary + paired comparison against the blend ----------
res = pd.DataFrame(results).round(2); res.to_csv(OUT, index=False)
print('\n================ FINAL TABLE ================'); print(res.drop(columns='Note', errors='ignore').to_string(index=False))
if 'RF + MLP Blend (final)' in preds:
    b = np.abs(y_test - preds['RF + MLP Blend (final)']); print('\nPaired difference in absolute error vs the blend (positive = blend better):')
    for k, v in preds.items():
        if k != 'RF + MLP Blend (final)':
            d = np.abs(y_test - v) - b; se = d.std() / np.sqrt(len(d))
            print(f'  {k:42s} {d.mean():+.3f}  (95% CI {d.mean()-1.96*se:+.3f} to {d.mean()+1.96*se:+.3f})')
print(f'\nSaved to {OUT}')
try:                                            # also copy to Google Drive if it is mounted
    import shutil; shutil.copy(OUT, '/content/drive/MyDrive/' + OUT)
except Exception: pass

rows 6325 | train 5060 | test 1265   (thesis: 6325 | 5060 | 1265)
>> Naive baseline (avg of last 4 same slots) MAE 3.293  RMSE 4.194  MAPE 31.83  WAPE 21.98  Peak 5.37
   (0s)
>> Reference: avg of last 12 same slots MAE 3.120  RMSE 3.956  MAPE 31.73  WAPE 20.83  Peak 5.18
   (0s)
>> Random Forest (alone)              MAE 3.090  RMSE 3.935  MAPE 30.50  WAPE 20.63  Peak 6.15
   (1248s)
   tuned Neural Network: {'mlpregressor__alpha': 0.01, 'mlpregressor__hidden_layer_sizes': (100,), 'mlpregressor__learning_rate_init': 0.01}  (71s)
>> Neural Network (alone, tuned)      MAE 3.157  RMSE 4.027  MAPE 32.10  WAPE 21.07  Peak 6.16
   (9s)
   tuned LightGBM: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 200, 'num_leaves': 15}  (20s)
>> LightGBM (tuned)                   MAE 3.095  RMSE 3.939  MAPE 30.76  WAPE 20.66  Peak 5.94
   (2s)
   tuned CatBoost: {'depth': 4, 'iterations': 300, 'learning_rate': 0.03}  (43s)
>> CatBoost (tuned)                   MAE 3.069  RMSE 3.915  MAPE 30.95  

## 6. Final feature list (18 features) and chronological 80/20 split

In [ ]:
# Multi-window volume lags (2, 4, 8, 12 weeks) — replaces single lag_8wk_avg
for weeks in [2, 4, 8, 12]:
    base[f'lag_{weeks}wk_avg'] = (
        base.groupby(['day_of_week','hour_slot'])['patient_count']
        .transform(lambda x: x.shift(1).rolling(weeks, min_periods=1).mean())
    )

feature_cols = [
    'hour_slot', 'day_of_week', 'month', 'week_of_year', 'is_weekend',
    'lag_2wk_avg', 'lag_4wk_avg', 'lag_8wk_avg', 'lag_12wk_avg',
    'lag_8wk_severity', 'lag_8wk_high_acuity', 'daily_cumulative_avg',
    'lag_8wk_avg_age', 'lag_8wk_avg_PainGrade', 'lag_8wk_avg_NeedFastExecute', 'lag_8wk_avg_CriticalStatus',
    'lag_8wk_Medical/Internal', 'lag_8wk_Trauma/Injury'
]
print(f'Total features: {len(feature_cols)}')

split_idx = int(len(base) * 0.8)
train = base.iloc[:split_idx].dropna(subset=feature_cols)
test = base.iloc[split_idx:].dropna(subset=feature_cols)

print(f'Train: {len(train)} rows, {train["date"].min()} to {train["date"].max()}')
print(f'Test: {len(test)} rows, {test["date"].min()} to {test["date"].max()}')

X_train = train[feature_cols]
y_train = train['patient_count']
X_test = test[feature_cols]
y_test = test['patient_count']

Total features: 18
Train: 5051 rows, 2017-03-28 00:00:00 to 2019-07-21 00:00:00
Test: 1274 rows, 2019-07-22 00:00:00 to 2020-02-21 00:00:00


## 7. Evaluate via walk-forward (rolling-origin) validation
Retrains weekly using real data only — confirmed to improve MAPE over a single static fit.

In [ ]:
def walk_forward_eval(data, feature_cols, target_col, initial_train_frac=0.8, step_size=42):
    data = data.sort_values(['date', 'hour_slot']).reset_index(drop=True)
    data = data.dropna(subset=feature_cols).reset_index(drop=True)
    n = len(data)
    train_end = int(n * initial_train_frac)

    all_actuals, all_preds = [], []
    current_train_end = train_end
    while current_train_end < n:
        train_data = data.iloc[:current_train_end]
        step_end = min(current_train_end + step_size, n)
        test_step = data.iloc[current_train_end:step_end]

        X_tr, y_tr = train_data[feature_cols], train_data[target_col]
        X_te, y_te = test_step[feature_cols], test_step[target_col]

        rf_wf = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42)
        rf_wf.fit(X_tr, y_tr)
        mlp_wf = MLPRegressor(hidden_layer_sizes=(50,), alpha=0.0001, learning_rate_init=0.01, max_iter=2000, random_state=42)
        mlp_wf.fit(X_tr, y_tr)

        preds = (rf_wf.predict(X_te) + mlp_wf.predict(X_te)) / 2
        all_actuals.extend(y_te.values)
        all_preds.extend(preds)
        current_train_end = step_end

    all_actuals, all_preds = np.array(all_actuals), np.array(all_preds)
    mae = mean_absolute_error(all_actuals, all_preds)
    rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
    mape = np.mean(np.abs((all_actuals - all_preds) / all_actuals)) * 100
    wape = np.sum(np.abs(all_actuals - all_preds)) / np.sum(all_actuals) * 100
    peak_thresh = np.quantile(all_actuals, 0.9)
    peak_mask = all_actuals >= peak_thresh
    peak_mae = mean_absolute_error(all_actuals[peak_mask], all_preds[peak_mask])

    print(f'Walk-forward (18 features) — MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%, WAPE: {wape:.2f}%, Peak MAE: {peak_mae:.2f}')

walk_forward_eval(base, feature_cols, 'patient_count')

## 8. Train final deployable model (on ALL data — this is what gets saved)
For deployment, we train one final model using every real row available (train+test combined), matching the spirit of walk-forward: always use all real data up to now.

In [ ]:
X_all = base.dropna(subset=feature_cols)[feature_cols]
y_all = base.dropna(subset=feature_cols)['patient_count']

rf_final = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42)
rf_final.fit(X_all, y_all)

mlp_final = MLPRegressor(hidden_layer_sizes=(50,), alpha=0.0001, learning_rate_init=0.01, max_iter=2000, random_state=42)
mlp_final.fit(X_all, y_all)

print('Final deployable model trained on all available real data.')

## 9. Save model and historical data for the backend

In [ ]:
final_models = {'rf': rf_final, 'mlp': mlp_final}
joblib.dump(final_models, 'flow_prediction_model.pkl')

base.dropna(subset=feature_cols)[['date','day_of_week','hour_slot'] + feature_cols[5:] + ['patient_count']].to_csv('historical_flow.csv', index=False)

print('Saved: flow_prediction_model.pkl, historical_flow.csv')

In [ ]:
from google.colab import files
files.download('flow_prediction_model.pkl')
files.download('historical_flow.csv')